In [1]:
import pandas as pd
import numpy as np

In [2]:
dia = pd.read_csv('Diabetes_train.csv') #Loading the train data

In [3]:
dia.shape #700k rows and 26 columns

(700000, 26)

In [4]:
dia.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 700000 entries, 0 to 699999
Data columns (total 26 columns):
 #   Column                              Non-Null Count   Dtype  
---  ------                              --------------   -----  
 0   id                                  700000 non-null  int64  
 1   age                                 700000 non-null  int64  
 2   alcohol_consumption_per_week        700000 non-null  int64  
 3   physical_activity_minutes_per_week  700000 non-null  int64  
 4   diet_score                          700000 non-null  float64
 5   sleep_hours_per_day                 700000 non-null  float64
 6   screen_time_hours_per_day           700000 non-null  float64
 7   bmi                                 700000 non-null  float64
 8   waist_to_hip_ratio                  700000 non-null  float64
 9   systolic_bp                         700000 non-null  int64  
 10  diastolic_bp                        700000 non-null  int64  
 11  heart_rate                

In [5]:
dia.sample()

,id,age,alcohol_consumption_per_week,physical_activity_minutes_per_week,diet_score,sleep_hours_per_day,screen_time_hours_per_day,bmi,waist_to_hip_ratio,systolic_bp,...,gender,ethnicity,education_level,income_level,smoking_status,employment_status,family_history_diabetes,hypertension_history,cardiovascular_history,diagnosed_diabetes
571262,571262,59,3,224,5.0,7.8,10.0,25.9,0.87,112,...,Male,White,Highschool,Middle,Never,Retired,0,0,0,0.0


In [6]:
dia.isna().sum() #Checking for missing values

,0
id,0
age,0
alcohol_consumption_per_week,0
physical_activity_minutes_per_week,0
diet_score,0
sleep_hours_per_day,0
screen_time_hours_per_day,0
bmi,0
waist_to_hip_ratio,0
systolic_bp,0


In [7]:
dia.drop(columns = ['id'], inplace = True) #dropping the id column because its not useful for prediction


In [9]:
dia['income_level'].value_counts() #ordinal encode
dia['smoking_status'].value_counts() #one hot encode
dia['education_level'].value_counts() #ordinal encode
dia['ethnicity'].value_counts() #one hot encode
dia['gender'].value_counts()#one hot encode
dia['employment_status'].value_counts() #one hot encode

,count
employment_status,
Employed,516170
Retired,115735
Unemployed,49787
Student,18308


In [10]:
from sklearn.model_selection import train_test_split, cross_val_score
X_train, X_test, y_train, y_test = train_test_split(dia.iloc[:, :24], dia.iloc[:, [-1]], random_state = 42, test_size = 0.2)
#spliting the data into train and test pairs to avoid data leakage

In [11]:
from sklearn.preprocessing import OrdinalEncoder, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import MinMaxScaler, StandardScaler


In [12]:
tnf1 = ColumnTransformer(transformers = [
    ('oe_income_level_edu_level',OrdinalEncoder(categories=[
    ['Low', 'Lower-Middle', 'Middle', 'Upper-Middle', 'High'],
    ['No formal', 'Highschool', 'Graduate', 'Postgraduate']]), [18, 17]),
    ('ohe_employment_status_gender_ethnicity_smoking_status', OneHotEncoder(drop = 'first', sparse_output = False), [20, 15, 16,19] )
    ], remainder = 'passthrough') #Encoding Catagorical data

In [41]:
from sklearn.preprocessing import PowerTransformer
tnf2 = ColumnTransformer(transformers = [
    ('yeo_johnson_0to14', PowerTransformer(), slice(0,15)),
    ('yeo_johnson_21to23', PowerTransformer(), slice(21,24))
    ], remainder = 'passthrough') #Used Yeo-Johnson transformation to stabilize skewed predictors and improve optimization.

In [13]:
tnf3 = ColumnTransformer([
    ('scale_age_systolic_bp_doastolic_bp_heart_rate', StandardScaler(), [0,8,9,10])
], remainder = 'passthrough' ) #scaling

In [ ]:
tnf4 = LogisticRegression(C = np.float64(1.0476157527896652))

In [137]:
tnf4 = LogisticRegression(max_iter=5000) #big dataset

In [138]:
pipe = Pipeline([('tnf1', tnf1),('tnf2', tnf2),('tnf3', tnf3),('tnf4', tnf4)])

In [139]:
pipe.fit(X_train, y_train.values.ravel()) #changing y_train column vector to 1d array to avoid warning

Pipeline(steps=[('tnf1',
                 ColumnTransformer(remainder='passthrough',
                                   transformers=[('oe_income_level_edu_level',
                                                  OrdinalEncoder(categories=[['Low',
                                                                              'Lower-Middle',
                                                                              'Middle',
                                                                              'Upper-Middle',
                                                                              'High'],
                                                                             ['No '
                                                                              'formal',
                                                                              'Highschool',
                                                                              'Graduate',
                                                                              'Postgraduate']]),
                                                  [18, 17]),
                                                 ('ohe_employment_status_gender_ethnicity_smoking_status',
                                                  OneHotEncoder(drop='first',
                                                                sparse_outp...
                 ColumnTransformer(remainder='passthrough',
                                   transformers=[('yeo_johnson_0to14',
                                                  PowerTransformer(),
                                                  slice(0, 15, None)),
                                                 ('yeo_johnson_21to23',
                                                  PowerTransformer(),
                                                  slice(21, 24, None))])),
                ('tnf3',
                 ColumnTransformer(remainder='passthrough',
                                   transformers=[('scale_age_systolic_bp_doastolic_bp_heart_rate',
                                                  StandardScaler(),
                                                  [0, 8, 9, 10])])),
                ('tnf4', LogisticRegression(max_iter=5000))])

In [140]:
y_pred = pipe.predict(X_test) #array of 0 and 1's
y_prob = pipe.predict_proba(X_test)[:, 1] #array of probabilities



In [141]:
from sklearn.metrics import accuracy_score
accuracy_score(y_test, y_pred) #Calculating model accuracy

0.6616428571428571

In [119]:
dia['diagnosed_diabetes'].value_counts()

,count
diagnosed_diabetes,
1.0,436307
0.0,263693


**So the actual dataset is imbalanced(more diabetics than non-diabetics)**

In [120]:
from sklearn.metrics import confusion_matrix #Calculating the confusion matrix that will give us the table of FPs and FNs
confusion_matrix(y_test, y_pred)  #FP = 35550, FN = 11851


array([[17079, 35550],
       [11851, 75520]])

In [113]:
from sklearn.metrics import classification_report
print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

         0.0       0.59      0.32      0.42     52629
         1.0       0.68      0.86      0.76     87371

    accuracy                           0.66    140000
   macro avg       0.64      0.59      0.59    140000
weighted avg       0.65      0.66      0.63    140000



* The dataset is imbalanced, with a higher proportion of diabetic cases.so the logistic regression model is naturally biased toward predicting the +ve class.

* the model prioritizes recall for diabetics (86%), successfully minimizing false negatives, but at the cost of increased false positives.(35550)

* Accuracy alone penalizes this behavior and fails to capture the underlying error trade-offs.



In [111]:
from sklearn.metrics import roc_auc_score #Calculating roc-auc
roc_auc_score(y_test, y_prob)


np.float64(0.6941634050175931)

* So the model assigns a higher probability to the diabetic ~69.4% of the time

In [130]:
y_pred_new = (y_prob >= 0.55).astype(int)  #increased the threshold value from 0.5 to 0.55

In [131]:
from sklearn.metrics import classification_report
print(classification_report(y_test, y_pred_new))


              precision    recall  f1-score   support

         0.0       0.55      0.46      0.50     52629
         1.0       0.71      0.78      0.74     87371

    accuracy                           0.66    140000
   macro avg       0.63      0.62      0.62    140000
weighted avg       0.65      0.66      0.65    140000



* **at the same Accuracy level 0.66, the error structure changed significantly.**

* ROC–AUC (~0.69) confirms that the model has moderate class-separation ability independent of threshold choice.

* By increasing the decision threshold to 0.55, precision and specificity improve while overall accuracy remains unchanged,  hence the threshold selection, not model accuracy, is governing the real-world decision behavior

In [142]:
from sklearn.model_selection import cross_val_score
cross_val_score(pipe, X_train, y_train.values.ravel(), cv = 5, scoring = 'roc_auc').mean() #cross validation

np.float64(0.6944129287908976)